In [24]:
import pandas as pd 
df=pd.read_csv('data/Churn_Modelling.csv')
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [25]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [26]:
df=pd.get_dummies(df, columns=['Geography', 'Gender'],drop_first=True,dtype=int)
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [27]:
df.rename(columns={'Geography_Spain':'IsSpain','Geography_Germany':'IsGermany','Gender_Male':'IsMale'}, inplace=True)
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,IsGermany,IsSpain,IsMale
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [28]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [29]:
from sklearn.model_selection import train_test_split
X=df.drop('Exited', axis=1)
y=df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
y_train.value_counts()

Exited
0    6370
1    1630
Name: count, dtype: int64

In [33]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report
import numpy as np

def ann(X_train, y_train, X_test, y_test):
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model=keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
      
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_scaled, y_train, epochs=20, batch_size=32, validation_data=(X_test_scaled, y_test))
    
    print(model.evaluate(X_test_scaled, y_test))
    
    y_pred=model.predict(X_test_scaled)
    y_pred=np.round(y_pred)
    print(classification_report(y_test, y_pred))
    
    return y_pred

In [34]:
y_pred = ann(X_train, y_train, X_test, y_test)

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-20 21:32:50.310441: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-03-20 21:32:50.310643: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-20 21:32:50.310660: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-20 21:32:50.311127: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-20 21:32:50.311147: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow 

Epoch 1/20


2026-03-20 21:32:51.897483: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-20 21:32:51.902979: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.7521 - loss: 0.5336 - val_accuracy: 0.8050 - val_loss: 0.4535
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8046 - loss: 0.4457 - val_accuracy: 0.8160 - val_loss: 0.4270
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8104 - loss: 0.4392 - val_accuracy: 0.8170 - val_loss: 0.4178
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8261 - loss: 0.4058 - val_accuracy: 0.8335 - val_loss: 0.3940
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8385 - loss: 0.3857 - val_accuracy: 0.8520 - val_loss: 0.3702
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8494 - loss: 0.3708 - val_accuracy: 0.8540 - val_loss: 0.3626
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8578 - loss: 0.3443 - val_accuracy: 0.8580 - val_loss: 0.3511
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8509 - loss: 0.3586 - val_accuracy: 0.858

2026-03-20 21:33:57.016060: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1593
           1       0.76      0.47      0.58       407

    accuracy                           0.86      2000
   macro avg       0.82      0.72      0.75      2000
weighted avg       0.85      0.86      0.85      2000



# Under Sampling

In [37]:
# Class count
count_class_0, count_class_1 = df.Exited.value_counts()
print(count_class_0, count_class_1)

7963 2037


In [38]:
# Divide by class
df_class_0 = df[df['Exited'] == 0]
df_class_1 = df[df['Exited'] == 1]

In [39]:
df_class_0.shape, df_class_1.shape

((7963, 12), (2037, 12))

In [41]:
df_class_0_under = df_class_0.sample(count_class_1)
df_under = pd.concat([df_class_0_under, df_class_1], axis=0)
df_under.Exited.value_counts()

Exited
0    2037
1    2037
Name: count, dtype: int64

In [42]:
X=df_under.drop('Exited', axis=1)
y=df_under['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
y_train.value_counts()

Exited
1    1630
0    1629
Name: count, dtype: int64

In [43]:
y_pred = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-20 21:43:06.699132: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.5987 - loss: 0.6738 - val_accuracy: 0.6626 - val_loss: 0.6313
Epoch 2/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6704 - loss: 0.6116 - val_accuracy: 0.6736 - val_loss: 0.6122
Epoch 3/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6839 - loss: 0.5919 - val_accuracy: 0.6810 - val_loss: 0.5973
Epoch 4/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6984 - loss: 0.5688 - val_accuracy: 0.6761 - val_loss: 0.5912
Epoch 5/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7161 - loss: 0.5491 - val_accuracy: 0.6945 - val_loss: 0.5795
Epoch 6/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7091 - loss: 0.5550 - val_accuracy: 0.7067 - val_loss: 0.5686
Epoch 7/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7435 - loss: 0.5337 - val_accuracy: 0.7190 - val_loss: 0.5517
Epoch 8/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7325 - loss: 0.5284 - val_accuracy: 0.725

# OverSampling

In [45]:
# Oversample 1-class and concat the DataFrames of both classes
df_class_1_over = df_class_1.sample(count_class_0, replace=True)
df_test_over = pd.concat([df_class_0, df_class_1_over], axis=0)

print('Random over-sampling:')
print(df_test_over.Exited.value_counts())

Random over-sampling:
Exited
0    7963
1    7963
Name: count, dtype: int64


In [46]:
X=df_test_over.drop('Exited', axis=1)
y=df_test_over['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
y_train.value_counts()

Exited
0    6370
1    6370
Name: count, dtype: int64

In [47]:
y_pred = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-20 21:47:03.536412: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


399/399 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.6250 - loss: 0.6455 - val_accuracy: 0.6943 - val_loss: 0.5789
Epoch 2/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7051 - loss: 0.5684 - val_accuracy: 0.7335 - val_loss: 0.5383
Epoch 3/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.7453 - loss: 0.5218 - val_accuracy: 0.7599 - val_loss: 0.4982
Epoch 4/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.7662 - loss: 0.4897 - val_accuracy: 0.7649 - val_loss: 0.4793
Epoch 5/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7676 - loss: 0.4816 - val_accuracy: 0.7728 - val_loss: 0.4770
Epoch 6/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7643 - loss: 0.4776 - val_accuracy: 0.7718 - val_loss: 0.4696
Epoch 7/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7727 - loss: 0.4710 - val_accuracy: 0.7762 - val_loss: 0.4655
Epoch 8/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7793 - loss: 0.4656 - val_accuracy: 0.765

2026-03-20 21:48:45.632850: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
              precision    recall  f1-score   support

           0       0.80      0.78      0.79      1593
           1       0.78      0.81      0.79      1593

    accuracy                           0.79      3186
   macro avg       0.79      0.79      0.79      3186
weighted avg       0.79      0.79      0.79      3186



# Smote

In [48]:
X=df.drop('Exited', axis=1)
y=df['Exited']

In [51]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(sampling_strategy='minority')
X_sm, y_sm=smote.fit_resample(X,y)
y_sm.value_counts()

/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Exited
1    7963
0    7963
Name: count, dtype: int64

In [52]:
X_test,X_train,y_test,y_train=train_test_split(X_sm,y_sm,test_size=0.2,random_state=42,stratify=y_sm)
y_train.value_counts()

Exited
0    1593
1    1593
Name: count, dtype: int64

In [53]:
y_pred = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-20 21:55:36.955987: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.6587 - loss: 0.6492 - val_accuracy: 0.7490 - val_loss: 0.5414
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7485 - loss: 0.5319 - val_accuracy: 0.7666 - val_loss: 0.4978
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7621 - loss: 0.5138 - val_accuracy: 0.7673 - val_loss: 0.4861
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.7610 - loss: 0.4948 - val_accuracy: 0.7776 - val_loss: 0.4716
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7812 - loss: 0.4744 - val_accuracy: 0.7827 - val_loss: 0.4649
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7781 - loss: 0.4796 - val_accuracy: 0.7818 - val_loss: 0.4637
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7817 - loss: 0.4639 - val_accuracy: 0.7811 - val_loss: 0.4648
Epoch 8/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.7752 - loss: 0.4685 - val_accuracy: 0.786

2026-03-20 21:56:40.125514: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
              precision    recall  f1-score   support

           0       0.76      0.88      0.81      6370
           1       0.86      0.72      0.78      6370

    accuracy                           0.80     12740
   macro avg       0.81      0.80      0.80     12740
weighted avg       0.81      0.80      0.80     12740



# Ensemble Method 

In [54]:
df.Exited.value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [55]:
X=df.drop('Exited', axis=1)
y=df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
y_train.value_counts()

Exited
0    6370
1    1630
Name: count, dtype: int64

In [56]:
df2=X_train.copy()
df2['Exited']=y_train
df2

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,IsGermany,IsSpain,IsMale,Exited
2151,753,57,7,0.00,1,1,0,159475.08,0,0,1,1
8392,739,32,3,102128.27,1,1,0,63981.37,1,0,1,1
5006,755,37,0,113865.23,2,1,1,117396.25,1,0,0,0
4117,561,37,5,0.00,2,1,0,83093.25,0,0,1,0
7182,692,49,6,110540.43,2,0,1,107472.99,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4555,688,35,6,0.00,1,1,0,25488.43,0,1,0,1
4644,712,74,5,0.00,2,0,0,151425.82,0,1,1,0
8942,667,37,9,71786.90,2,1,1,67734.79,0,0,0,0
2935,687,35,8,100988.39,2,1,0,22247.27,0,1,1,0


In [57]:
df2_class_0 = df2[df2['Exited'] == 0]
df2_class_1 = df2[df2['Exited'] == 1]

In [58]:
def get_train_batch(majority_class, minority_class,start,end):
    df_train=pd.concat([majority_class[start:end], minority_class], axis=0)
    X_train=df_train.drop('Exited', axis=1)
    y_train=df_train['Exited'] 
    return X_train, y_train


In [59]:
X_train,y_train=get_train_batch(df2_class_0, df2_class_1, 0, 1630)
y_pred1 = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-21 01:44:04.271897: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5583 - loss: 0.6857 - val_accuracy: 0.6040 - val_loss: 0.6601
Epoch 2/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6682 - loss: 0.6236 - val_accuracy: 0.7015 - val_loss: 0.5778
Epoch 3/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6816 - loss: 0.5961 - val_accuracy: 0.6870 - val_loss: 0.5920
Epoch 4/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6853 - loss: 0.5855 - val_accuracy: 0.6420 - val_loss: 0.6365
Epoch 5/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6894 - loss: 0.5898 - val_accuracy: 0.7435 - val_loss: 0.5258
Epoch 6/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7177 - loss: 0.5655 - val_accuracy: 0.7505 - val_loss: 0.5199
Epoch 7/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7316 - loss: 0.5475 - val_accuracy: 0.7695 - val_loss: 0.4811
Epoch 8/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7354 - loss: 0.5297 - val_accuracy: 0.741

In [60]:
X_train,y_train=get_train_batch(df2_class_0, df2_class_1, 1630, 3260)
y_pred2 = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5595 - loss: 0.6851 - val_accuracy: 0.6215 - val_loss: 0.6588
Epoch 2/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6492 - loss: 0.6309 - val_accuracy: 0.7015 - val_loss: 0.5881
Epoch 3/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6771 - loss: 0.5989 - val_accuracy: 0.6930 - val_loss: 0.5906
Epoch 4/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6918 - loss: 0.5830 - val_accuracy: 0.6855 - val_loss: 0.5941
Epoch 5/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6991 - loss: 0.5665 - val_accuracy: 0.6895 - val_loss: 0.5940
Epoch 6/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7223 - loss: 0.5441 - val_accuracy: 0.7300 - val_loss: 0.5444
Epoch 7/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7267 - loss: 0.5306 - val_accuracy: 0.7390 - val_loss: 0.5370
Epoch 8/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7603 - loss: 0.4969 - val_accuracy: 0.787

2026-03-21 01:45:33.010823: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


In [61]:
X_train,y_train=get_train_batch(df2_class_0, df2_class_1, 3260, 4890)
y_pred3 = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


102/102 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5524 - loss: 0.6828 - val_accuracy: 0.6420 - val_loss: 0.6641
Epoch 2/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6533 - loss: 0.6255 - val_accuracy: 0.7445 - val_loss: 0.5639
Epoch 3/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6905 - loss: 0.5985 - val_accuracy: 0.7110 - val_loss: 0.5886
Epoch 4/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7171 - loss: 0.5637 - val_accuracy: 0.7390 - val_loss: 0.5589
Epoch 5/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7011 - loss: 0.5764 - val_accuracy: 0.7250 - val_loss: 0.5665
Epoch 6/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7238 - loss: 0.5494 - val_accuracy: 0.6455 - val_loss: 0.6467
Epoch 7/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7284 - loss: 0.5417 - val_accuracy: 0.6950 - val_loss: 0.5905
Epoch 8/20
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7428 - loss: 0.5224 - val_accuracy: 0.707

In [62]:
X_train,y_train=get_train_batch(df2_class_0, df2_class_1, 4890, 6370)
y_pred4 = ann(X_train, y_train, X_test, y_test)

Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.5826 - loss: 0.6740 - val_accuracy: 0.6650 - val_loss: 0.6377
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6488 - loss: 0.6244 - val_accuracy: 0.5895 - val_loss: 0.6946
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6746 - loss: 0.5972 - val_accuracy: 0.7300 - val_loss: 0.5526
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7006 - loss: 0.5735 - val_accuracy: 0.7150 - val_loss: 0.5624
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7184 - loss: 0.5581 - val_accuracy: 0.7080 - val_loss: 0.5661
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7107 - loss: 0.5505 - val_accuracy: 0.7650 - val_loss: 0.5001
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7311 - loss: 0.5398 - val_accuracy: 0.7655 - val_loss: 0.4978
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7373 - loss: 0.5233 - val_accuracy: 0.7715 - val_loss: 0.

2026-03-21 01:46:39.323952: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


In [63]:
len(y_pred1)

2000

In [70]:
y_pred_final = y_pred1.copy()
for i in range(len(y_pred1)):
    n_ones = y_pred1[i] + y_pred2[i] + y_pred3[i] + y_pred4[i]
    if n_ones>=3:
        y_pred_final[i] = 1
    else:
        y_pred_final[i] = 0


In [71]:
cl_rep = classification_report(y_test, y_pred_final)
print(cl_rep)

              precision    recall  f1-score   support

           0       0.92      0.81      0.86      1593
           1       0.48      0.71      0.58       407

    accuracy                           0.79      2000
   macro avg       0.70      0.76      0.72      2000
weighted avg       0.83      0.79      0.80      2000

